# 2단계(v2): 낙상 클래스 오류 분석 (고친 데이터셋 기준)

1단계에서 저장한 임베딩을 드라이브에서 읽어 쓰기 때문에, 임베딩을 다시 뽑을 필요가 없습니다. (Colab을 껐다 켜도 괜찮습니다.)

**실행 전에 할 일**: PC의 `LAB\upload_v2` 폴더에 있는 `manifest_v2_train.csv`와 `manifest_v2_val.csv`를 `4_dataset_v2.zip`과 **같은 드라이브 폴더**에 올려 주세요. (용량이 작습니다.)

위에서부터 순서대로 실행하고, **3번, 4번, 5번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기 · 읽기

In [ ]:
import glob, os
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 같은 드라이브 폴더에 올렸는지 확인하세요."
    return hits[0]

d = np.load(find("embeddings_v2.npz"))
X, S, clip_idx = d["X"], d["speech"], d["clip_idx"]

df = pd.read_csv(find("dataset_v2.csv"), encoding="utf-8-sig")
n0 = len(df)
df["id"] = df["path"].str.extract(r"([^/]+)\.wav$")[0]
man = pd.concat([pd.read_csv(find(f), encoding="utf-8-sig")
                 for f in ["manifest_v2_train.csv", "manifest_v2_val.csv"]])
df = df.merge(man[["id", "lr_corr", "orig_channels"]], on="id", how="left")
assert len(df) == n0, "manifest의 id가 겹쳐서 행이 늘어났어요"
print("클립", n0, "개 / manifest와 짝이 안 맞는 클립:", int(df["orig_channels"].isna().sum()))

df["speech_max"] = pd.Series(S.astype(np.float32)).groupby(clip_idx).max().reindex(range(n0)).values

## 2. 같은 베이스라인을 다시 학습 (1단계와 같은 방식)

In [ ]:
from sklearn.linear_model import LogisticRegression

starts = np.r_[0, np.flatnonzero(np.diff(clip_idx)) + 1]
assert len(starts) == len(df)
M = np.add.reduceat(X.astype(np.float32), starts, axis=0) / np.diff(np.r_[starts, len(X)])[:, None]

y = (df["label"] == "urgent").astype(int).values
tr = (df["split"] == "train").values
clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(M[tr], y[tr])
df["correct"] = clf.predict(M) == y

## 3. 학습 데이터에서도 낙상을 못 맞히나?
학습 데이터 점수는 높은데 Validation만 낮다면 "낙상이 원래 어렵다"가 아니라 **Training과 Validation의 조건이 달라서**일 가능성이 큽니다.

In [ ]:
print(df.groupby(["class", "split"])["correct"].mean().unstack().round(3)
        .rename(columns={"train": "학습 데이터", "val": "Validation"}))

## 4. Validation 낙상은 어떤 클립에서 틀리나?

In [ ]:
g = df[(df["class"] == "10.낙상") & (df["split"] == "val")].copy()
g["위상"] = np.where(g["lr_corr"].isna(), "모노/해당없음",
                   np.where(g["lr_corr"] < 0, "위상 반대(수정함)", "같은 위상"))
g["말소리"] = np.where(g["speech_max"] >= 0.5, "있음(>=0.5)", "없음")

for by in ["위상", "orig_channels", "distance", "place", "말소리", "generation"]:
    print(f"\n[낙상 Validation] {by}별 맞힌 비율")
    print(g.groupby(by)["correct"].agg(개수="size", 맞힌비율="mean").round(3))

cat = g.groupby("category")["correct"].agg(개수="size", 맞힌비율="mean")
print("\n맞힌 비율이 낮은 세부 분류 (3개 이상인 것만):")
print(cat[cat["개수"] >= 3].sort_values("맞힌비율").head(8).round(3))

## 5. 낙상 Training과 Validation은 어떻게 다른가?

In [ ]:
rows = []
for sp in ["train", "val"]:
    s = df[(df["class"] == "10.낙상") & (df["split"] == sp)]
    rows.append({"split": sp, "개수": len(s),
                 "스테레오 비율": (s["orig_channels"] == 2).mean(),
                 "위상 반대 비율": (s["lr_corr"] < 0).mean(),
                 "음량 중앙(dBFS)": s["rms_dbfs"].median(),
                 "근접 비율": (s["distance"] == "근접").mean(),
                 "실내 비율": (s["place"] == "실내").mean()})
print(pd.DataFrame(rows).set_index("split").round(2))

tr_cat = set(df[(df["class"] == "10.낙상") & (df["split"] == "train")]["category"])
va_cat = df[(df["class"] == "10.낙상") & (df["split"] == "val")]["category"]
print("\nValidation 낙상 중 Training에 없는 세부 분류 비율:", round(float((~va_cat.isin(tr_cat)).mean()), 2))